# FLAN-T5 LoRA fine-tuning for text simplification in Colab

This notebook trains a FLAN-T5-base model with LoRA on a JSON dataset containing `text` and `simplified` fields.
It uses a 70:30 train/test split, saves the LoRA adapter, and also saves a merged full model for inference.

In [ ]:
!pip install -q transformers datasets peft accelerate sentencepiece scikit-learn

In [ ]:
!pip install -q --upgrade "torchao>=0.16.0" transformers datasets peft accelerate sentencepiece scikit-learn
print('Upgraded torchao and core training packages. Restart the Colab runtime before running the notebook imports.')

In [ ]:
import os
import json
import torch
from google.colab import drive, files
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, TaskType

drive.mount('/content/drive')

In [ ]:
# Upload your JSON file from your computer when prompted.
# If you already saved it in Google Drive, replace the path below.

json_path = None

candidate_paths = [
    '/content/drive/MyDrive/simplified_1.json',
    '/content/drive/MyDrive/combined_simplified_19830.json',
]

for path in candidate_paths:
    if os.path.exists(path):
        json_path = path
        break

if json_path is None:
    uploaded = files.upload()
    json_path = '/content/' + next(iter(uploaded))

print('Using file:', json_path)

In [ ]:
with open(json_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

if isinstance(raw_data, dict):
    records = raw_data.get('data') or raw_data.get('items') or []
else:
    records = raw_data

normalized = []
for item in records:
    if isinstance(item, dict):
        text = item.get('text') or item.get('source') or item.get('input')
        target = item.get('simplified') or item.get('target') or item.get('output')
        if text and target:
            normalized.append({'text': str(text), 'simplified': str(target)})

print('Number of usable rows:', len(normalized))
if len(normalized) == 0:
    raise ValueError('No valid rows were found. Make sure your JSON has text/simplified fields.')

train_records, test_records = train_test_split(normalized, test_size=0.3, random_state=42)

train_dataset = Dataset.from_list(train_records)
test_dataset = Dataset.from_list(test_records)
dataset = DatasetDict({'train': train_dataset, 'test': test_dataset})
dataset

In [ ]:
model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print('Using device:', device)
print('Model:', model_name)

In [ ]:
def preprocess_function(examples):
    inputs = [f'Simplify the following text:\n{txt}' for txt in examples['text']]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(text_target=examples['simplified'], max_length=256, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=['text', 'simplified'])

print('Train examples:', len(tokenized_dataset['train']))
print('Test examples:', len(tokenized_dataset['test']))
print('Sample tokenized train example:')
print(tokenized_dataset['train'][0])
print('Non-pad label count in sample:', sum(1 for v in tokenized_dataset['train'][0]['labels'] if v != -100))

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q', 'v'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir='/content/drive/MyDrive/flan_t5_lora_simplify',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    eval_strategy='epoch',
    predict_with_generate=True,
    fp16=False,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator,
)

trainer.train()

In [ ]:
import os
import torch
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftConfig, PeftModel

adapter_dir = '/content/drive/MyDrive/flan_t5_lora_adapter'
merged_dir = '/content/drive/MyDrive/flan_t5_lora_merged'
output_dir = '/content/drive/MyDrive/flan_t5_lora_simplify'
drive_root = '/content/drive'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

drive.mount(drive_root, force_remount=True)
print('Drive mounted:', os.path.exists(drive_root))
print('MyDrive exists:', os.path.exists('/content/drive/MyDrive'))
print('default output_dir exists:', os.path.exists(output_dir))
if os.path.exists(output_dir):
    print('default output_dir contents:', sorted(os.listdir(output_dir)))


def find_adapter_dirs(root):
    found = []
    for dirpath, dirs, files in os.walk(root):
        if 'adapter_config.json' in files:
            found.append(dirpath)
    return sorted(found)

adapter_dirs = find_adapter_dirs(drive_root)
print('adapter dirs found under drive root:', adapter_dirs[:20])

if not adapter_dirs:
    raise FileNotFoundError(
        'No PEFT adapter folders were found under /content/drive.\n'
        'Please verify Drive is mounted and that your adapter folder contains adapter_config.json.'
    )

adapter_dir = adapter_dirs[-1]
print('Using adapter directory:', adapter_dir)
print('adapter dir contents:', sorted(os.listdir(adapter_dir)))

if 'model' not in globals() or 'tokenizer' not in globals():
    model_name = 'google/flan-t5-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    base_model.config.tie_word_embeddings = False
    base_model.to(device)

    peft_config = PeftConfig.from_pretrained(adapter_dir, local_files_only=True)
    model = PeftModel.from_pretrained(
        base_model,
        adapter_dir,
        peft_config=peft_config,
        local_files_only=True,
    )
    model.to(device)
else:
    print('Using existing in-memory model/tokenizer.')

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('Adapter saved to:', adapter_dir)

merged_model = model.merge_and_unload()
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print('Merged full model saved to:', merged_dir)


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

adapter_dir = '/content/drive/MyDrive/flan_t5_lora_adapter'
merged_dir = '/content/drive/MyDrive/flan_t5_lora_merged'
output_dir = '/content/drive/MyDrive/flan_t5_lora_simplify'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if 'model' not in globals() or 'tokenizer' not in globals():
    model_name = 'google/flan-t5-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    base_model.to(device)

    checkpoint_dirs = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')] if os.path.exists(output_dir) else []
    if checkpoint_dirs:
        checkpoint_numbers = [int(d.split('-')[-1]) for d in checkpoint_dirs]
        latest_ckpt = max(checkpoint_numbers)
        ckpt_path = os.path.join(output_dir, f'checkpoint-{latest_ckpt}')
        print('Loading LoRA adapter from checkpoint:', ckpt_path)
        model = PeftModel.from_pretrained(base_model, ckpt_path)
        model.to(device)
    else:
        raise FileNotFoundError('No training checkpoint was found. Run training again or point output_dir to the correct checkpoint folder.')
else:
    print('Using existing in-memory model/tokenizer.')


def simplify_text(text, use_merged=True):
    prompt = f'Simplify the following text:\n{text}'
    inputs = tokenizer(prompt, return_tensors='pt', max_length=512, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    if use_merged:
        loaded_model = AutoModelForSeq2SeqLM.from_pretrained(merged_dir).to(device)
    else:
        loaded_model = model.to(device)
    outputs = loaded_model.generate(**inputs, max_new_tokens=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

sample_text = normalized[0]['text']
print(simplify_text(sample_text, use_merged=True))
